In [ ]:
import os
import re
import json
import time
import base64
import mimetypes
import requests

from datetime import datetime, timezone
from dotenv import load_dotenv

try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.json import JSON as RichJSON
    HAS_RICH = True
except ImportError:
    HAS_RICH = False

load_dotenv()

BASE_URL = os.getenv("DERMA_API_URL", "").strip().rstrip("/")
EMAIL = os.getenv("DERMA_EMAIL", "").strip()
PASSWORD = os.getenv("DERMA_PASSWORD", "")
API_KEY = os.getenv("DERMA_API_KEY", "").strip()
IMAGE_PATH = os.getenv("SAMPLE_IMAGE_PATH", "")

POLL_INTERVAL = 1
MAX_WAIT_SECONDS = 180

UUID_REGEX = re.compile(
    r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b"
)

if not IMAGE_PATH or not os.path.isfile(IMAGE_PATH):
    raise FileNotFoundError(f"Nie znaleziono zdjęcia: {IMAGE_PATH}")

console = Console() if HAS_RICH else None
session = requests.Session()
session.headers.update({"Accept": "application/json"})
if API_KEY:
    session.headers.update({"X-API-Key": API_KEY})


def log(msg, style="cyan"):
    ts = datetime.now().strftime("%H:%M:%S")
    if HAS_RICH:
        console.print(f"[dim][{ts}][/dim] [{style}]{msg}[/{style}]")
    else:
        print(f"[{ts}] {msg}")


def show_analysis(title, data, border_style="blue"):
    dumped = json.dumps(data, indent=2, ensure_ascii=False)
    if HAS_RICH:
        console.print(
            Panel(
                RichJSON(dumped),
                title=f"[bold {border_style}]{title}[/bold {border_style}]",
                border_style=border_style,
                expand=False,
            )
        )
    else:
        line = "=" * 60
        print(f"\n{line}\n{title}\n{line}\n{dumped}\n{line}\n")


def norm_id(val):
    if not val:
        return None
    m = UUID_REGEX.search(str(val).strip())
    return m.group(0).lower() if m else str(val).strip().lower()


def decode_jwt(token):
    try:
        payload = token.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        return json.loads(base64.urlsafe_b64decode(payload))
    except Exception:
        return {}


def get_user_id(claims):
    for k in [
        "nameid",
        "sub",
        "userId",
        "user_id",
        "id",
        "http://schemas.xmlsoap.org/ws/2005/05/identity/claims/nameidentifier",
    ]:
        if claims.get(k):
            return norm_id(claims[k])
    return None


def get_list(data):
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for k in ["items", "Items", "data", "Data", "value", "Value", "results", "Results"]:
            if isinstance(data.get(k), list):
                return data[k]
    return []


def extract_id(obj, response=None):
    if isinstance(obj, str):
        return norm_id(obj)
    if isinstance(obj, dict):
        for k in ["id", "Id", "analysisId", "AnalysisId"]:
            if obj.get(k):
                return norm_id(obj[k])
        for k in ["data", "Data", "value", "Value", "analysis", "Analysis"]:
            if isinstance(obj.get(k), dict):
                found = extract_id(obj[k])
                if found:
                    return found
    if response is not None:
        loc = response.headers.get("Location")
        if loc and norm_id(loc):
            return norm_id(loc)
        if response.text and norm_id(response.text):
            return norm_id(response.text)
    return None


def fetch_analyses():
    r = session.get(f"{BASE_URL}/api/SkinAnalysis", timeout=30)
    r.raise_for_status()
    return get_list(r.json() if r.text.strip() else [])


def fetch_analysis_by_id(analysis_id):
    r = session.get(f"{BASE_URL}/api/SkinAnalysis/{analysis_id}", timeout=30)
    if r.status_code == 200 and r.text.strip():
        return r.json()
    return None


def get_status(obj):
    return obj.get("status", obj.get("Status")) if isinstance(obj, dict) else None

# Obsługa statusów liczbowych i tekstowych
def status_value(value):
    if value is None:
        return None
    try:
        return int(value)
    except (ValueError, TypeError):
        return str(value).strip()


def fetch_analysis_results():
    r = session.get(
        f"{BASE_URL}/api/AnalysisResult",
        timeout=30
    )
    
    r.raise_for_status()

    return get_list(r.json() if r.text.strip() else [])


def fetch_acne_result_by_analysis_id(skin_analysis_id):
    results = fetch_analysis_results()

    for result in results:
        result_analysis_id = norm_id(
            result.get("skinAnalysisId")
            or result.get("SkinAnalysisId")
        )

        if result_analysis_id == norm_id(skin_analysis_id):
            return result

    return None


In [ ]:
# 1. Logowanie
login_r = session.post(f"{BASE_URL}/api/Account/login", json={"Email": EMAIL, "Password": PASSWORD}, timeout=30)
login_r.raise_for_status()
login_json = login_r.json()
token = login_json.get("token") or login_json.get("Token") or login_json.get("accessToken")
if not token:
    raise RuntimeError("Brak tokenu JWT.")
session.headers.update({"Authorization": f"Bearer {token}"})

user_id = get_user_id(decode_jwt(token))
log(f"Zalogowano (userId={user_id})", style="green")

# 2. Lista przed dodaniem
before_list = fetch_analyses()
before_ids = {norm_id(extract_id(a)) for a in before_list if extract_id(a)}

# 3. Dodanie analizy
form_data = {
    "Notes": "Test",
    "UploadDate": datetime.now(timezone.utc).isoformat(),
    "Tags": ["3", "4", "2"],
}
mime_type = mimetypes.guess_type(IMAGE_PATH)[0] or "application/octet-stream"

with open(IMAGE_PATH, "rb") as f:
    create_r = session.post(
        f"{BASE_URL}/api/SkinAnalysis",
        data=form_data,
        files={"Image": (os.path.basename(IMAGE_PATH), f, mime_type)},
        timeout=30,
    )
create_r.raise_for_status()

create_json = create_r.json() if create_r.text.strip() else None
post_id = extract_id(create_json, create_r)
log(f"POST zwrócił ID: {post_id}")

# 4. Znalezienie analizy i wyświetlenie od razu (przed zmianą statusu)
time.sleep(2)
after_list = fetch_analyses()
diff_items = [a for a in after_list if norm_id(extract_id(a)) not in before_ids]

analysis = next((a for a in after_list if norm_id(extract_id(a)) == post_id), None)
if analysis is None and diff_items:
    analysis = diff_items[0]
    log(f"Uwaga: ID z POST nie ma na liście, używam nowego ID z listy: {extract_id(analysis)}", style="yellow")

if analysis is None:
    raise RuntimeError("Nie udało się znaleźć nowej analizy na liście.")

analysis_id = extract_id(analysis)
initial_status = get_status(analysis)

show_analysis(
    f"PRZED ZMIANĄ STATUSU | id={analysis_id} | status={initial_status}",
    analysis,
    border_style="yellow",
)


# 5. Monitorowanie zmian statusu 0 -> 1 -> 2 lub 3

current = analysis
current_status = get_status(current)


initial_status = status_value(current_status)

log(f"STATUS POCZĄTKOWY: {initial_status}", style="yellow")

# Pomiar czasu całego procesu i poszczególnych etapów
process_start = time.monotonic()
stage_start = process_start

stage_times = {}

last_status = initial_status

while time.monotonic() - process_start < MAX_WAIT_SECONDS:

    time.sleep(POLL_INTERVAL)

    by_id = fetch_analysis_by_id(analysis_id)

    if by_id is not None:
        current = by_id
    else:
        lst = fetch_analyses()
        found = next(
            (a for a in lst if norm_id(extract_id(a)) == analysis_id),
            None
        )
        if found is not None:
            current = found

    new_status = status_value(get_status(current))

    if new_status != last_status:

        now = time.monotonic()
        stage_elapsed = now - stage_start
        total_elapsed = now - process_start

        transition = f"{last_status} -> {new_status}"

        stage_times[transition] = stage_elapsed

        log(
            f"ZMIANA STATUSU: {transition}",
            style="green"
        )

        log(
            f"Czas tego etapu: {stage_elapsed:.2f} s",
            style="cyan"
        )

        log(
            f"Czas całkowity: {total_elapsed:.2f} s",
            style="yellow"
        )

        last_status = new_status
        stage_start = now

        # Status 2 lub 3 oznacza zakończenie procesu
        if new_status in (2, 3):
            break

# Podsumowanie
total_time = time.monotonic() - process_start

print("\n" + "=" * 55)
print("PODSUMOWANIE CZASU ANALIZY")
print("=" * 55)

for transition, elapsed in stage_times.items():
    print(f"{transition}: {elapsed:.2f} s")

print("-" * 55)
print(f"CAŁKOWITY CZAS: {total_time:.2f} s")
print(f"KOŃCOWY STATUS: {last_status}")
print("=" * 55)

show_analysis(
    f"WYNIK ANALIZY | status={last_status} | id={analysis_id}",
    current,
    border_style="green" if last_status == 2 else "red",
)

In [ ]:
# 6. Oczekiwanie na wynik AnalysisResult

log("Oczekiwanie na wynik Acne AnalysisResult...", style="yellow")

result_start = time.monotonic()
acne_result = None

RESULT_POLL_INTERVAL = 1
RESULT_MAX_WAIT = 90

while time.monotonic() - result_start < RESULT_MAX_WAIT:

    acne_result = fetch_acne_result_by_analysis_id(analysis_id)

    if acne_result is not None:
        break

    elapsed = time.monotonic() - result_start

    log(
        f"Oczekiwanie na AnalysisResult... {elapsed:.1f}s",
        style="dim"
    )

    time.sleep(RESULT_POLL_INTERVAL)


result_elapsed = time.monotonic() - result_start

if acne_result is None:
    log(
        f"Nie znaleziono AnalysisResult po {result_elapsed:.2f}s",
        style="red"
    )

else:
    log(
        f"AnalysisResult znaleziony po {result_elapsed:.2f}s",
        style="green"
    )

    show_analysis(
        "ACNE ANALYSIS RESULT",
        acne_result,
        border_style="green"
    )

    # Szczegółowe podsumowanie
    detections = (
        acne_result.get("detections")
        or acne_result.get("Detections")
        or []
    )

    print("\n" + "=" * 60)
    print("PODSUMOWANIE ACNE")
    print("=" * 60)

    print(f"Analysis ID: {analysis_id}")
    print(f"Result ID: {acne_result.get('id')}")
    print(f"Acne Score: {acne_result.get('acneScore')}")
    print(f"Acne Count: {acne_result.get('acneCount')}")
    print(f"TTA Applied: {acne_result.get('tta_applied')}")
    print(f"SAHI Applied: {acne_result.get('sahi_applied')}")
    print(f"Detections: {len(detections)}")
    print(f"Output Image: {acne_result.get('outputImageUrl')}")

    print("\n" + "=" * 60)
    print(f"CZAS OCZEKIWANIA NA WYNIK: {result_elapsed:.2f}s")
    print("=" * 60)
